# Actual code

In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from sklearn.metrics import roc_auc_score
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

# ===================== Parameters (shared by BOTH analyses) =====================
variant_class       = 'missense'                    # defines variant filters + tool set
# selected_categories = ['missense', 'conservation', 'genetic_diversity', 'gnomad']  # tool categories compared in the scatter
selected_categories = ['missense', 'conservation', 'genetic_diversity', 'gnomad']

mac                 = 20                             # rare-variant cap (phenotype correlations)
only_snps           = True
exclude_clinvar     = False
only_clinvar        = False

# ClinVar pathogenicity labels
patho_labels  = ['Pathogenic', 'Likely_pathogenic']
benign_labels = ['Benign', 'Likely_benign']         # None -> all non-pathogenic treated as benign
MIN_PATHO_PER_GENE    = 3
MIN_VARIANTS_PER_GENE = 5

# ===================== Configs =====================
CFG = 'PATH_TO_FILE'
variant_class_config = yaml.safe_load(open(f'{CFG}/config_variant_classes.yaml'))
anno_config          = yaml.safe_load(open(f'{CFG}/config_correlations.yaml'))

anno_config_df = pl.DataFrame([
    {'category': cat, 'annotation': anno, 'color': p['color'],
     'label': p['label'], 'annotation_dir': p.get('direction', 1)}
    for cat, annos in anno_config['rare_variant_annotations'].items()
    for anno, p in annos.items()
]).with_columns(pl.col('annotation_dir').cast(pl.Int8))
all_annotation_list = anno_config_df['annotation'].to_list()

vc = variant_class_config[variant_class]
vc_filters = vc.get('variant_filtering') or []

# --- Variant annotations for the target genes ---
_dynamic_filters = [eval(f) for f in vc_filters]
if exclude_clinvar:
    _dynamic_filters.append(pl.col('clinical_significance').is_null())
elif only_clinvar:
    _dynamic_filters.append(pl.col('clinical_significance').is_not_null())
if only_snps:
    _dynamic_filters.append((pl.col('ref').str.len_chars() == 1) & (pl.col('alt').str.len_chars() == 1))

# Tools (annotations) evaluated in BOTH analyses
selected_annos = anno_config_df.filter(pl.col('category').is_in(selected_categories))['annotation'].to_list()

# ===================== Local data paths =====================
CLINVAR_PATH = 'PATH_TO_FILE'

print(f"variant_class = {variant_class}")
print(f"categories    = {selected_categories}")
print(f"{len(selected_annos)} tools: {selected_annos}")
anno_config_df.filter(pl.col('annotation').is_in(selected_annos))

In [ ]:
# ===================== ClinVar per-gene auROC =====================

def _label_mask(s, labels):
    m = pl.Series([False] * len(s))
    for lbl in labels:
        m = m | s.str.contains(lbl, literal=True).fill_null(False)
    return m

clinvar = (
    pl.scan_parquet(CLINVAR_PATH)
    .filter((pl.col('ref').str.len_chars() == 1) & (pl.col('alt').str.len_chars() == 1))
    .filter(*_dynamic_filters)
)

clin_annos = [a for a in selected_annos if a in clinvar.collect_schema().names()]
clinvar = clinvar.select(['id', 'region', 'clinical_significance'] + clin_annos).collect()

# Pathogenic / benign masks (handles compound labels like "Benign/Likely_benign")
clin_sig  = clinvar['clinical_significance'].fill_null('')
is_patho  = _label_mask(clin_sig, patho_labels)
is_benign = _label_mask(clin_sig, benign_labels) if benign_labels is not None else ~is_patho
clinvar   = clinvar.with_columns(is_patho=is_patho.cast(pl.Int8)).filter(is_patho | is_benign)

# Long format + direction correction (same melt+direction pattern used for the correlations)
clin_long = (
    clinvar
    .unpivot(index=['id', 'region', 'is_patho'], on=clin_annos,
             variable_name='annotation', value_name='raw')
    .drop_nulls('raw')
    .drop_nans('raw')
    .join(anno_config_df.select(['annotation', 'annotation_dir']), on='annotation', how='left')
    .with_columns(score = pl.col('raw').cast(pl.Float64) * pl.col('annotation_dir').cast(pl.Float64))
)

# Per-gene, per-tool auROC (pathogenic vs benign), with coverage thresholds
rows = []
for (region, annotation), g in clin_long.group_by(['region', 'annotation']):
    y = g['is_patho'].to_numpy()
    if len(y) < MIN_VARIANTS_PER_GENE or y.sum() < MIN_PATHO_PER_GENE:
    # if y.sum() < MIN_PATHO_PER_GENE:
        continue
    if y.sum() == 0 or y.sum() == len(y):
        continue
    rows.append({'region': region, 'annotation': annotation,
                 'auroc': float(roc_auc_score(y, g['score'].to_numpy())),
                 'n_patho': int(y.sum()), 'n_total': len(y)})

clinvar_auroc_df = pl.DataFrame(rows).join(
    anno_config_df.select(['annotation', 'label', 'color', 'category']), on='annotation', how='left'
)
print(f"ClinVar auROC: {clinvar_auroc_df.height} rows "
      f"({clinvar_auroc_df['region'].n_unique()} genes x {clinvar_auroc_df['annotation'].n_unique()} tools)")

clinvar_auroc_df.sort(['annotation', 'auroc'], descending=[False, True])

In [ ]:
import pandas as pd
import itertools
import heapq
import numpy as np
from scipy import stats

exclude_annos = []

# --- 1. Annotation set + label map ---
plot_annotations = clinvar_auroc_df.select("annotation").unique().to_series().to_list()
plot_annotations = [a for a in plot_annotations if a not in exclude_annos]
anno_to_label = dict(clinvar_auroc_df.select(["annotation", "label"]).unique().iter_rows())

# --- 2. Precompute every head-to-head paired comparison once ---
# diff[(a, b)] = mean(corr_beta_a - corr_beta_b) over genes/traits shared by a and b
# pval[{a, b}] = Wilcoxon signed-rank p-value (symmetric)
vals_by_ann = {
    a: clinvar_auroc_df.filter(pl.col("annotation") == a).select(["region", "auroc"])
    for a in plot_annotations
}
ALPHA = 0.05
diff, pval = {}, {}
for a, b in itertools.combinations(plot_annotations, 2):
    paired = vals_by_ann[a].join(vals_by_ann[b], on=["region"], suffix="_b")
    ca = paired["auroc"].to_numpy()
    cb = paired["auroc_b"].to_numpy()
    d = float(ca.mean() - cb.mean()) if len(ca) else 0.0
    if len(ca) >= 10 and not np.allclose(ca, cb):
        _, pv = stats.wilcoxon(ca, cb, alternative="two-sided")
    else:
        pv = 1.0
    diff[(a, b)], diff[(b, a)] = d, -d
    pval[frozenset((a, b))] = pv

def beats(a, b):  # a is SIGNIFICANTLY better than b (higher corr on shared genes)
    return pval[frozenset((a, b))] < ALPHA and diff[(a, b)] > 0

# --- 3. Dominance ordering (upset-safe topological sort) ---
wins      = {t: sum(beats(t, u) for u in plot_annotations if u != t) for t in plot_annotations}
losses    = {t: sum(beats(u, t) for u in plot_annotations if u != t) for t in plot_annotations}
advantage = {t: sum(diff[(t, u)] for u in plot_annotations if u != t) for t in plot_annotations}

def _key(t):
    return (losses[t], -wins[t], -advantage[t], t)   # trailing `t` = deterministic tiebreak

successors = {t: [u for u in plot_annotations if u != t and beats(t, u)] for t in plot_annotations}
indeg = dict(losses)          # in-degree in the full graph == total losses

heap = [(_key(t), t) for t in plot_annotations if indeg[t] == 0]
heapq.heapify(heap)

ordered_tools, placed = [], set()
while len(ordered_tools) < len(plot_annotations):
    if heap:
        _, t = heapq.heappop(heap)
        if t in placed:
            continue
    else:                                              # residual cycle -> force-break
        remaining = [x for x in plot_annotations if x not in placed]
        t = min(remaining, key=_key)
    ordered_tools.append(t)
    placed.add(t)
    for u in successors[t]:
        if u in placed:
            continue
        indeg[u] -= 1
        if indeg[u] == 0:
            heapq.heappush(heap, (_key(u), u))

ordered_labels = [anno_to_label.get(t, t) for t in ordered_tools]   # <-- add this

# --- 4. Build heatmap from the precomputed comparisons (no recompute) ---
heatmap_data = []
for ann_x, ann_y in itertools.product(ordered_tools, repeat=2):
    label_x = anno_to_label.get(ann_x, ann_x)
    label_y = anno_to_label.get(ann_y, ann_y)
    if ann_x == ann_y:
        heatmap_data.append({'Tool_X': label_x, 'Tool_Y': label_y, 'mean_diff': 0.0, 'sig': ''})
        continue
    md = diff[(ann_y, ann_x)]                  # mean(Tool_Y) - mean(Tool_X), matches original fill
    pv = pval[frozenset((ann_x, ann_y))]
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    heatmap_data.append({'Tool_X': label_x, 'Tool_Y': label_y, 'mean_diff': md, 'sig': sig})

df_heat = pd.DataFrame(heatmap_data)

# Lock in categorical order so best tools appear top/right
df_heat['Tool_X'] = pd.Categorical(df_heat['Tool_X'], categories=ordered_labels,       ordered=True)
df_heat['Tool_Y'] = pd.Categorical(df_heat['Tool_Y'], categories=ordered_labels[::-1], ordered=True)

n_tools = len(ordered_tools)

# --- 5. Plot ---
plot = (
    ggplot(
        df_heat, 
        aes(x='Tool_X', y='Tool_Y', fill='mean_diff'))
    + geom_tile(color="#FFFFFF", size=0.5)
    + geom_text(aes(label='sig'), color="black", size=12, va='center', nudge_y=-0.1)
    + scale_fill_gradient2(low="#2C7BB6", mid="#FFFFFF", high="#D7191C", midpoint=0)
    + labs(
        title=f"{variant_class} variants - ({clinvar_auroc_df['region'].n_unique()} assocs.)",
        x="Tool X",
        y="Tool Y",
        fill="Tool Y − X\n(avg auROC)"
    )
    + theme_minimal()
    + theme(
        figure_size=(n_tools * 0.6 + 2.5, n_tools * 0.6 + 1.5),
        aspect_ratio=1,
        axis_title=element_text(size=13),
        axis_text=element_text(size=13),
        axis_text_x=element_text(rotation=45, hjust=1),
        panel_grid=element_blank(),
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

FIG_DIR = "PATH_TO_FILE"
plot.save(f"{FIG_DIR}/F3_clinvar_auroc_heatmap.svg", dpi=200)

plot
